# Embedding + Vector Indexing Pipeline (Comments + Products)

هدف: امبد کردن ~۱۲۴ فایل parquet کامنت (~۶.۵ میلیون رکورد) و ۱۹ فایل parquet محصول، و ایندکس همزمان در **Milvus** (large-scale vector search) + **ChromaDB** (RAG-oriented retrieval) + بارگذاری متادیتای کامل در **PostgreSQL**.

### شِمای واقعی داده (طبق نمونه‌ای که فرستادید)

**کامنت‌ها**: `id, title, body, created_at, rate, recommendation_status, is_buyer, product_id, advantages, disadvantages, likes, dislikes, seller_title, seller_code, true_to_size_rate, raw_text, raw_text_normalized, aspects_json (رشته‌ی JSON), num_aspects`

**محصولات**: `id, title_fa, Rate, Rate_cnt, Category1, Category2, Brand, Price, Seller, Is_Fake, min_price_last_month, sub_category, raw_text, raw_text_normalized`

نکته‌ی مهم: هم کامنت‌ها و هم محصولات یک ستون `id` واقعی و یکتا دارن — پس دیگه لازم نیست id مصنوعی (`file#row_idx`) بسازیم؛ مستقیم از همون `id` واقعی به‌عنوان primary key در هر سه مقصد (Milvus/Chroma/Postgres) استفاده می‌کنیم. این باعث می‌شه بعداً join بین `comments_meta.product_id` و `products_meta.id` هم مستقیم و درست کار کنه.

`aspects_json` یک **رشته‌ی JSON** است (نه لیست پایتونی)، پس قبل از استفاده باید `json.loads` بشه.

هیچ ستون sentiment در سطح کل کامنت وجود نداره (فقط per-aspect `sentiment` داخل `aspects_json`)؛ به‌عنوان یک فیلد کمکیِ فیلترپذیر در Milvus، یک `sentiment_bucket` heuristic از روی `rate` و/یا میانگین sentiment اسپکت‌ها می‌سازیم — این محاسبه‌شده است، نه داده‌ی خام، و باید به همین چشم دیده بشه.

### معماری ذخیره‌سازی (بدون فشار به Drive محدودتون)
هیچ وکتوری روی Drive ذخیره نمی‌شه؛ فقط یک SQLite چک‌پوینت کوچیک برای resume در سطح فایل+بچ. همه‌ی upsertها بر اساس `id` واقعی idempotent هستن.

#### نصب پکیج‌ها

In [ ]:
!pip install -q -U sentence-transformers pymilvus chromadb psycopg2-binary tqdm pyarrow


#### Imports

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

import gc
import json
import math
import sqlite3
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

from pymilvus import connections, utility, FieldSchema, CollectionSchema, Collection, DataType

import chromadb

import psycopg2
from psycopg2.extras import execute_values, Json


#### Config

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---------- مدل امبدینگ ----------
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
USE_E5_PREFIXES = True   # مدل‌های E5 با پیشوند "query: " / "passage: " دقت بهتری می‌دن
EMBED_BATCH_SIZE = 128
MAX_SEQ_LEN = 256

# ---------- مسیرها (روی Drive - فقط ورودی + چک‌پوینت کوچیک) ----------
BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Cleaned_output")
COMMENTS_DIR = BASE_DIR / "digikala-comments_parts"   # ۱۲۴ فایل parquet کامنت
PRODUCTS_DIR = BASE_DIR / "digikala-products_parts"   # ۱۹ فایل parquet محصول -- TODO: اسم پوشه واقعی رو چک کنید

CKPT_PATH = str(BASE_DIR / "embedding_checkpoint.sqlite3")

# ---------- ستون‌های کامنت (طبق نمونه‌ی واقعی) ----------
COMMENT_ID_COL = "id"
COMMENT_TEXT_COL = "raw_text_normalized"
COMMENT_PRODUCT_ID_COL = "product_id"
COMMENT_RATE_COL = "rate"                          # 1..5
COMMENT_RECO_COL = "recommendation_status"         # recommended / not_recommended / no_idea
COMMENT_IS_BUYER_COL = "is_buyer"
COMMENT_ASPECTS_COL = "aspects_json"               # رشته‌ی JSON: [{"term":..,"sentiment":..,...}]

# ---------- ستون‌های محصول (طبق نمونه‌ی واقعی) ----------
PRODUCT_ID_COL = "id"
PRODUCT_TEXT_COLS = ["title_fa", "Category1", "Category2", "Brand", "sub_category"]
PRODUCT_CATEGORY_COL = "Category1"
PRODUCT_BRAND_COL = "Brand"
PRODUCT_PRICE_COL = "Price"

# ---------- سوییچ فعال/غیرفعال هر مقصد ----------
ENABLE_MILVUS = True
ENABLE_CHROMA = True
ENABLE_POSTGRES = True

# ---------- اتصال‌ها (حتماً پر کنید) ----------
MILVUS_URI = "http://YOUR_MILVUS_HOST:19530"
MILVUS_TOKEN = ""

CHROMA_HOST = "YOUR_CHROMA_HOST"
CHROMA_PORT = 8000
# اگه سرور ریموت Chroma ندارید (فقط تست، حجم مصرف می‌کنه):
# chroma_client = chromadb.PersistentClient(path=str(BASE_DIR / "chroma_local"))

POSTGRES_DSN = "dbname=YOUR_DB user=YOUR_USER password=YOUR_PASS host=YOUR_HOST port=5432"

MILVUS_COMMENTS_COLLECTION = "digikala_comments"
MILVUS_PRODUCTS_COLLECTION = "digikala_products"
CHROMA_COMMENTS_COLLECTION = "digikala_comments"
CHROMA_PRODUCTS_COLLECTION = "digikala_products"


#### Load Embedding Model

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LEN
EMBED_DIM = model.get_sentence_embedding_dimension()
print("embedding dim:", EMBED_DIM)


def embed_texts(texts, is_query=False):
    if USE_E5_PREFIXES:
        prefix = "query: " if is_query else "passage: "
        texts = [prefix + (t or "") for t in texts]
    vecs = model.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return vecs.astype(np.float32)


#### توابع کمکی برای اسپکت‌ها و sentiment

`aspects_json` یک رشته‌ی JSON هست؛ اینجا parse می‌کنیم و یک `sentiment_bucket` تقریبی (heuristic، نه داده‌ی خام) برای فیلتر کردن در Milvus می‌سازیم: اول از میانگین sentiment اسپکت‌ها، اگر اسپکتی نبود از `rate`.

In [ ]:
def parse_aspects_json(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    try:
        parsed = json.loads(raw) if isinstance(raw, str) else raw
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []


def aspects_to_terms_string(aspects):
    terms = sorted({a.get("term", "") for a in aspects if isinstance(a, dict) and a.get("term")})
    return ", ".join(terms)


def sentiment_from_rate(rate):
    try:
        r = int(rate)
    except (TypeError, ValueError):
        return "unknown"
    if r >= 4:
        return "positive"
    if r == 3:
        return "neutral"
    if r >= 1:
        return "negative"
    return "unknown"


def derive_sentiment_bucket(rate, aspects):
    """heuristic: میانگین sentiment اسپکت‌ها، وگرنه بر اساس rate."""
    sentiments = [a.get("sentiment") for a in aspects if isinstance(a, dict) and a.get("sentiment")]
    if sentiments:
        counts = Counter(sentiments)
        return counts.most_common(1)[0][0]
    return sentiment_from_rate(rate)


#### چک‌پوینت محلی (SQLite) — resume در سطح فایل + بچ

In [ ]:
def get_ckpt_conn():
    conn = sqlite3.connect(CKPT_PATH)
    conn.execute("""CREATE TABLE IF NOT EXISTS batches (
        source TEXT, file_name TEXT, batch_idx INTEGER,
        PRIMARY KEY (source, file_name, batch_idx)
    )""")
    conn.execute("""CREATE TABLE IF NOT EXISTS files_done (
        source TEXT, file_name TEXT,
        PRIMARY KEY (source, file_name)
    )""")
    conn.commit()
    return conn


def is_batch_done(conn, source, file_name, batch_idx):
    cur = conn.execute(
        "SELECT 1 FROM batches WHERE source=? AND file_name=? AND batch_idx=?",
        (source, file_name, batch_idx),
    )
    return cur.fetchone() is not None


def mark_batch_done(conn, source, file_name, batch_idx):
    conn.execute("INSERT OR IGNORE INTO batches VALUES (?,?,?)", (source, file_name, batch_idx))
    conn.commit()


def is_file_done(conn, source, file_name):
    cur = conn.execute("SELECT 1 FROM files_done WHERE source=? AND file_name=?", (source, file_name))
    return cur.fetchone() is not None


def mark_file_done(conn, source, file_name):
    conn.execute("INSERT OR IGNORE INTO files_done VALUES (?,?)", (source, file_name))
    conn.commit()


def progress_summary(conn, source):
    n_files = conn.execute("SELECT COUNT(*) FROM files_done WHERE source=?", (source,)).fetchone()[0]
    n_batches = conn.execute("SELECT COUNT(*) FROM batches WHERE source=?", (source,)).fetchone()[0]
    print(f"[{source}] فایل‌های کامل‌شده: {n_files}  |  بچ‌های کامل‌شده: {n_batches}")


#### اتصال به Milvus

In [ ]:
def connect_milvus():
    connections.connect(alias="default", uri=MILVUS_URI, token=MILVUS_TOKEN)
    print("Milvus connected:", MILVUS_URI)


def get_or_create_milvus_collection(name, dim, extra_fields=None):
    if utility.has_collection(name):
        coll = Collection(name)
        coll.load()
        return coll

    fields = [
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=64),
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=dim),
    ]
    if extra_fields:
        fields.extend(extra_fields)

    schema = CollectionSchema(fields=fields, description=f"{name} embeddings")
    coll = Collection(name=name, schema=schema)
    coll.create_index(
        field_name="vector",
        index_params={"index_type": "HNSW", "metric_type": "COSINE", "params": {"M": 16, "efConstruction": 200}},
    )
    coll.load()
    print(f"Milvus collection created: {name} (dim={dim})")
    return coll


if ENABLE_MILVUS:
    connect_milvus()

    comments_extra_fields = [
        FieldSchema(name="sentiment_bucket", dtype=DataType.VARCHAR, max_length=16),
        FieldSchema(name="product_id", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="rate", dtype=DataType.INT64),
        FieldSchema(name="is_buyer", dtype=DataType.BOOL),
    ]
    products_extra_fields = [
        FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=128),
        FieldSchema(name="brand", dtype=DataType.VARCHAR, max_length=128),
    ]

    milvus_comments = get_or_create_milvus_collection(MILVUS_COMMENTS_COLLECTION, EMBED_DIM, comments_extra_fields)
    milvus_products = get_or_create_milvus_collection(MILVUS_PRODUCTS_COLLECTION, EMBED_DIM, products_extra_fields)


#### اتصال به ChromaDB

In [ ]:
def connect_chroma():
    client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    print("Chroma connected:", CHROMA_HOST, CHROMA_PORT)
    return client

    # اگه سرور ریموت ندارید (فقط تست، حجم مصرف می‌کنه):
    # return chromadb.PersistentClient(path=str(BASE_DIR / "chroma_local"))


if ENABLE_CHROMA:
    chroma_client = connect_chroma()
    chroma_comments = chroma_client.get_or_create_collection(
        name=CHROMA_COMMENTS_COLLECTION, metadata={"hnsw:space": "cosine"}
    )
    chroma_products = chroma_client.get_or_create_collection(
        name=CHROMA_PRODUCTS_COLLECTION, metadata={"hnsw:space": "cosine"}
    )


#### اتصال به PostgreSQL

In [ ]:
def get_pg_conn():
    return psycopg2.connect(POSTGRES_DSN)


def ensure_pg_tables(conn):
    with conn.cursor() as cur:
        cur.execute("""
        CREATE TABLE IF NOT EXISTS comments_meta (
            id VARCHAR(64) PRIMARY KEY,
            product_id VARCHAR(64),
            title TEXT,
            body TEXT,
            rate SMALLINT,
            recommendation_status VARCHAR(32),
            is_buyer BOOLEAN,
            aspects_json JSONB,
            source_file VARCHAR(255)
        );
        """)
        cur.execute("""
        CREATE TABLE IF NOT EXISTS products_meta (
            id VARCHAR(64) PRIMARY KEY,
            title_fa TEXT,
            category1 VARCHAR(255),
            category2 VARCHAR(255),
            brand VARCHAR(255),
            price BIGINT,
            source_file VARCHAR(255)
        );
        """)
    conn.commit()


def upsert_comments_meta(conn, rows):
    sql = """
    INSERT INTO comments_meta (id, product_id, title, body, rate, recommendation_status, is_buyer, aspects_json, source_file)
    VALUES %s
    ON CONFLICT (id) DO UPDATE SET
        product_id = EXCLUDED.product_id,
        title = EXCLUDED.title,
        body = EXCLUDED.body,
        rate = EXCLUDED.rate,
        recommendation_status = EXCLUDED.recommendation_status,
        is_buyer = EXCLUDED.is_buyer,
        aspects_json = EXCLUDED.aspects_json,
        source_file = EXCLUDED.source_file;
    """
    with conn.cursor() as cur:
        execute_values(cur, sql, rows)
    conn.commit()


def upsert_products_meta(conn, rows):
    sql = """
    INSERT INTO products_meta (id, title_fa, category1, category2, brand, price, source_file)
    VALUES %s
    ON CONFLICT (id) DO UPDATE SET
        title_fa = EXCLUDED.title_fa,
        category1 = EXCLUDED.category1,
        category2 = EXCLUDED.category2,
        brand = EXCLUDED.brand,
        price = EXCLUDED.price,
        source_file = EXCLUDED.source_file;
    """
    with conn.cursor() as cur:
        execute_values(cur, sql, rows)
    conn.commit()


pg_conn = None
if ENABLE_POSTGRES:
    pg_conn = get_pg_conn()
    ensure_pg_tables(pg_conn)
    print("Postgres connected & tables ensured.")


#### Pipeline کامنت‌ها

از `id` واقعی هر کامنت به‌عنوان primary key در هر سه مقصد استفاده می‌شه، پس resume/retry همیشه idempotent هست و نیازی به id مصنوعی نیست.

In [ ]:
def run_comments_pipeline():
    files = sorted(COMMENTS_DIR.glob("*.parquet"))
    print(f"Found {len(files)} comment files")
    ckpt = get_ckpt_conn()

    for file in files:
        fname = file.name
        if is_file_done(ckpt, "comments", fname):
            print(f"[comments] skip (already done): {fname}")
            continue

        df = pd.read_parquet(file)
        n = len(df)
        n_batches = math.ceil(n / EMBED_BATCH_SIZE)

        for b in tqdm(range(n_batches), desc=f"comments:{fname}"):
            if is_batch_done(ckpt, "comments", fname, b):
                continue

            chunk = df.iloc[b * EMBED_BATCH_SIZE : (b + 1) * EMBED_BATCH_SIZE]
            records = chunk.to_dict("records")

            ids = [str(r[COMMENT_ID_COL]) for r in records]
            texts = [str(r.get(COMMENT_TEXT_COL) or "") for r in records]
            product_ids = [str(r.get(COMMENT_PRODUCT_ID_COL, "unknown")) for r in records]
            rates = [int(r[COMMENT_RATE_COL]) if pd.notna(r.get(COMMENT_RATE_COL)) else 0 for r in records]
            is_buyers = [bool(r.get(COMMENT_IS_BUYER_COL, False)) for r in records]
            aspects_lists = [parse_aspects_json(r.get(COMMENT_ASPECTS_COL)) for r in records]
            sentiment_buckets = [derive_sentiment_bucket(r.get(COMMENT_RATE_COL), a) for r, a in zip(records, aspects_lists)]

            embeddings = embed_texts(texts, is_query=False)

            if ENABLE_MILVUS:
                milvus_comments.upsert([ids, embeddings.tolist(), sentiment_buckets, product_ids, rates, is_buyers])

            if ENABLE_CHROMA:
                metadatas = [
                    {
                        "product_id": pid,
                        "rate": rate,
                        "recommendation_status": str(r.get(COMMENT_RECO_COL) or "unknown"),
                        "is_buyer": buyer,
                        "sentiment_bucket": sb,
                        "aspects": aspects_to_terms_string(asp),
                        "source_file": fname,
                    }
                    for r, pid, rate, buyer, sb, asp in zip(records, product_ids, rates, is_buyers, sentiment_buckets, aspects_lists)
                ]
                chroma_comments.upsert(
                    ids=ids,
                    embeddings=embeddings.tolist(),
                    documents=[t[:2000] for t in texts],
                    metadatas=metadatas,
                )

            if ENABLE_POSTGRES and pg_conn is not None:
                pg_rows = [
                    (
                        ids[i],
                        product_ids[i],
                        str(records[i].get("title") or ""),
                        str(records[i].get("body") or ""),
                        rates[i],
                        str(records[i].get(COMMENT_RECO_COL) or "unknown"),
                        is_buyers[i],
                        Json(aspects_lists[i]),
                        fname,
                    )
                    for i in range(len(records))
                ]
                upsert_comments_meta(pg_conn, pg_rows)

            mark_batch_done(ckpt, "comments", fname, b)

        if ENABLE_MILVUS:
            milvus_comments.flush()
        mark_file_done(ckpt, "comments", fname)
        progress_summary(ckpt, "comments")

        del df
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    ckpt.close()
    print("\n✅ Comments pipeline finished.")


run_comments_pipeline()


#### تست نمونه — Similarity Search روی کامنت‌ها

In [ ]:
sample_query = "کیفیت عطر و ماندگاری بوش خیلی خوب بود"
q_vec = embed_texts([sample_query], is_query=True)[0].tolist()

if ENABLE_MILVUS:
    print("=== Milvus results ===")
    res = milvus_comments.search(
        data=[q_vec],
        anns_field="vector",
        param={"metric_type": "COSINE", "params": {"ef": 64}},
        limit=5,
        output_fields=["sentiment_bucket", "product_id", "rate"],
    )
    for hit in res[0]:
        print(f"id={hit.id}  score={hit.distance:.4f}  sentiment={hit.entity.get('sentiment_bucket')}  "
              f"product_id={hit.entity.get('product_id')}  rate={hit.entity.get('rate')}")

if ENABLE_CHROMA:
    print("\n=== Chroma results ===")
    res = chroma_comments.query(query_embeddings=[q_vec], n_results=5)
    for i in range(len(res["ids"][0])):
        print(f"id={res['ids'][0][i]}  dist={res['distances'][0][i]:.4f}  doc={res['documents'][0][i][:80]!r}")


#### Discovery — بررسی شِمای واقعی فایل‌های محصول (پیشنهاد: قبل از اجرای pipeline محصولات یک‌بار اجرا کنید)

In [ ]:
product_files = sorted(PRODUCTS_DIR.glob("*.parquet"))
print(f"Found {len(product_files)} product files")

if product_files:
    sample_df = pd.read_parquet(product_files[0])
    print("Columns:", list(sample_df.columns))
    print(sample_df.head(3))
else:
    print("⚠️ هیچ فایلی پیدا نشد — PRODUCTS_DIR رو در سلول Config چک کنید.")


#### Pipeline محصولات

In [ ]:
def build_product_text(row):
    parts = [str(row.get(c, "") or "") for c in PRODUCT_TEXT_COLS]
    return " | ".join(p for p in parts if p and p != "nan")


def run_products_pipeline():
    files = sorted(PRODUCTS_DIR.glob("*.parquet"))
    print(f"Found {len(files)} product files")
    ckpt = get_ckpt_conn()

    for file in files:
        fname = file.name
        if is_file_done(ckpt, "products", fname):
            print(f"[products] skip (already done): {fname}")
            continue

        df = pd.read_parquet(file)
        n = len(df)
        n_batches = math.ceil(n / EMBED_BATCH_SIZE)

        for b in tqdm(range(n_batches), desc=f"products:{fname}"):
            if is_batch_done(ckpt, "products", fname, b):
                continue

            chunk = df.iloc[b * EMBED_BATCH_SIZE : (b + 1) * EMBED_BATCH_SIZE]
            records = chunk.to_dict("records")

            ids = [str(r[PRODUCT_ID_COL]) for r in records]
            texts = [build_product_text(r) for r in records]
            categories = [str(r.get(PRODUCT_CATEGORY_COL) or "unknown") for r in records]
            categories2 = [str(r.get("Category2") or "") for r in records]
            brands = [str(r.get(PRODUCT_BRAND_COL) or "unknown") for r in records]
            prices = [int(r[PRODUCT_PRICE_COL]) if pd.notna(r.get(PRODUCT_PRICE_COL)) else 0 for r in records]
            titles = [str(r.get("title_fa") or "") for r in records]

            embeddings = embed_texts(texts, is_query=False)

            if ENABLE_MILVUS:
                milvus_products.upsert([ids, embeddings.tolist(), categories, brands])

            if ENABLE_CHROMA:
                metadatas = [
                    {"category": c, "brand": b, "price": p, "source_file": fname}
                    for c, b, p in zip(categories, brands, prices)
                ]
                chroma_products.upsert(
                    ids=ids,
                    embeddings=embeddings.tolist(),
                    documents=[t[:2000] for t in texts],
                    metadatas=metadatas,
                )

            if ENABLE_POSTGRES and pg_conn is not None:
                pg_rows = list(zip(ids, titles, categories, categories2, brands, prices, [fname] * len(ids)))
                upsert_products_meta(pg_conn, pg_rows)

            mark_batch_done(ckpt, "products", fname, b)

        if ENABLE_MILVUS:
            milvus_products.flush()
        mark_file_done(ckpt, "products", fname)
        progress_summary(ckpt, "products")

        del df
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    ckpt.close()
    print("\n✅ Products pipeline finished.")


run_products_pipeline()


#### تست نمونه — Similarity Search روی محصولات

In [ ]:
sample_product_query = "عروسک کودک اردک"
qp_vec = embed_texts([sample_product_query], is_query=True)[0].tolist()

if ENABLE_MILVUS:
    print("=== Milvus results ===")
    res = milvus_products.search(
        data=[qp_vec],
        anns_field="vector",
        param={"metric_type": "COSINE", "params": {"ef": 64}},
        limit=5,
        output_fields=["category", "brand"],
    )
    for hit in res[0]:
        print(f"id={hit.id}  score={hit.distance:.4f}  category={hit.entity.get('category')}  brand={hit.entity.get('brand')}")

if ENABLE_CHROMA:
    print("\n=== Chroma results ===")
    res = chroma_products.query(query_embeddings=[qp_vec], n_results=5)
    for i in range(len(res["ids"][0])):
        print(f"id={res['ids'][0][i]}  dist={res['distances'][0][i]:.4f}  doc={res['documents'][0][i][:80]!r}")


#### Progress Dashboard

هر وقت خواستید ببینید تا کجا پیش رفته (مثلاً بعد از یک قطعی و از سرگیری)، همین سلول رو اجرا کنید.

In [ ]:
ckpt = get_ckpt_conn()
progress_summary(ckpt, "comments")
progress_summary(ckpt, "products")
ckpt.close()

if ENABLE_MILVUS:
    print("\nMilvus comments count:", milvus_comments.num_entities)
    print("Milvus products count:", milvus_products.num_entities)

if ENABLE_CHROMA:
    print("\nChroma comments count:", chroma_comments.count())
    print("Chroma products count:", chroma_products.count())
